In [1]:
import requests
import pandas as pd
import time
import math

def generar_panel_genes():
    genes_recolectados = []
    
    # 1. CLINGEN
    print("ClinGen")
    try:
        tabla_clingen = pd.read_csv("Clingen-Gene-Disease-Summary-2026-02-13.csv", skiprows=4)
        palabras_clave = 'thromb|coagul|bleed|platelet|hemosta|blood|embol|fibrin|plasmin|leiden|von willebrand|myeloproliferative|hemochromatosis|homocystein|vascular|endothel|erythrocyt|polycythemia'
        candidatos = tabla_clingen['DISEASE LABEL'].astype(str).str.contains(palabras_clave, case=False, na=False)
        tabla_filtrada = tabla_clingen[candidatos]

        for indice, fila in tabla_filtrada.iterrows():
            gen = str(fila['GENE SYMBOL'])
            clasificacion = str(fila['CLASSIFICATION'])
            
            if gen == 'nan' or not gen.strip():
                continue

            if clasificacion in ['Definitive', 'Strong']:
                genes_recolectados.append({'Gen': gen, 'Fuente': 'ClinGen', 'Prioridad': 1})
            elif clasificacion in ['Moderate', 'Limited']:
                genes_recolectados.append({'Gen': gen, 'Fuente': 'ClinGen', 'Prioridad': 2})
            elif clasificacion in ['Disputed', 'Refuted', 'No Known Disease Relationship']:
                genes_recolectados.append({'Gen': gen, 'Fuente': 'ClinGen', 'Prioridad': 3})

        print(f"    Añadidos {len(tabla_filtrada)} posibles genes de ClinGen.")
        
    except Exception as e:
        print(f"  Error leyendo ClinGen")


    # 2. PANELAPP
    print("\nPanelApp")
    paneles = ["516", "175", "545"]

    for panel_id in paneles:
        url = f"https://panelapp.genomicsengland.co.uk/api/v1/panels/{panel_id}/"
        try:
            respuesta = requests.get(url, timeout=15)
            if respuesta.status_code == 200:
                datos = respuesta.json()
                lista_genes_panel = datos.get('genes', [])
                
                for info_gen in lista_genes_panel:
                    simbolo_gen = info_gen['gene_data']['gene_symbol']
                    nivel_confianza = info_gen.get('confidence_level')
                    
                    if nivel_confianza == '3':
                        genes_recolectados.append({'Gen': simbolo_gen, 'Fuente': f'PanelApp_{panel_id}', 'Prioridad': 1})
                    elif nivel_confianza == '2':
                        genes_recolectados.append({'Gen': simbolo_gen, 'Fuente': f'PanelApp_{panel_id}', 'Prioridad': 2})
                        
                print(f"    Panel {panel_id} descargado.")
            else:
                print(f"    Error HTTP {respuesta.status_code} al bajar PanelApp {panel_id}")
        except Exception as e:
            print(f"  Error conectando a PanelApp {panel_id}: {e}")


    # 3. QUICKGO
    print("\nQuickGO")
    terminos_go = ["GO:0007599", "GO:0007596"]
    
    for termino in terminos_go:
        pagina_actual = 1
        total_paginas = 1 
        
        while pagina_actual <= total_paginas:
            parametros = {'goId': termino, 'taxonId': '9606', 'limit': 100, 'page': pagina_actual}
            url = "https://www.ebi.ac.uk/QuickGO/services/annotation/search"
            
            try:
                respuesta = requests.get(url, params=parametros, headers={'Accept': 'application/json'}, timeout=30)
                if respuesta.status_code != 200: 
                    break 
                
                datos = respuesta.json()
                resultados = datos.get('results', [])
                
                total_resultados = datos.get('pageInfo', {}).get('total', 1)
                total_paginas = math.ceil(total_resultados / 100.0)
                
                for resultado in resultados:
                    if 'symbol' in resultado:
                        simbolo_gen = resultado['symbol']
                        genes_recolectados.append({'Gen': simbolo_gen, 'Fuente': f'QuickGO_{termino}', 'Prioridad': 4})
                
                pagina_actual += 1
                time.sleep(0.5)

            except Exception as e:
                print(f"    Error conectando con QuickGO ({termino}): {e}")
                break
                
        print(f"    Término {termino} descargado.")


    # LIMPIEZA
    if len(genes_recolectados) > 0:
        tabla_final = pd.DataFrame(genes_recolectados)
        tabla_final = tabla_final.dropna(subset=['Gen'])
        
        # Ordenar y borrar duplicados
        tabla_final = tabla_final.sort_values('Prioridad').drop_duplicates(subset='Gen', keep='first')
        
        # Guardar en CSV
        tabla_final.to_csv("panel_genes.csv", index=False)
        
        # Guardar en TXT para que SnpSift lo pueda leer
        with open("panel_genes.txt", "w") as f:
            f.write("\n".join(tabla_final['Gen'].astype(str).tolist()))
        
        print("\n=== RESUMEN ===")
        print(f"Total de genes únicos guardados: {len(tabla_final)}")
        print(f"  Prioridad 1 (ClinGen Definitive/PanelApp): {len(tabla_final[tabla_final['Prioridad']==1])}")
        print(f"  Prioridad 2 (ClinGen Moderate/PanelApp): {len(tabla_final[tabla_final['Prioridad']==2])}")
        print(f"  Prioridad 3 (ClinGen Disputado): {len(tabla_final[tabla_final['Prioridad']==3])}")
        print(f"  Prioridad 4 (QuickGO Ontología): {len(tabla_final[tabla_final['Prioridad']==4])}")
    else:
        print("No se ha encontrado ningún gen.")

if __name__ == "__main__":
    generar_panel_genes()

Procesando archivo local de ClinGen...
    ClinGen procesado. Añadidos 86 posibles genes.

Descargando paneles de PanelApp...
    Panel 516 descargado.
    Panel 175 descargado.
    Panel 545 descargado.

Genes en la ontología QuickGO...
    Término GO:0007599 explorado por completo.
    Término GO:0007596 explorado por completo.

=== RESUMEN ===
Total de genes únicos guardados: 202
  Prioridad 1 (ClinGen Definitive/PanelApp): 118
  Prioridad 2 (ClinGen Moderate/PanelApp): 30
  Prioridad 3 (ClinGen Disputado): 1
  Prioridad 4 (QuickGO Ontología): 53


In [3]:
with open('panel_genes.txt', 'r') as f:
     genes_panel = set(line.strip().upper() for line in f if line.strip())

df = pd.read_excel('excel_isth_2024.xlsx')
    
col = 'Gene symbol (HGNC)'
df.columns = [str(c).strip() for c in df.columns]
genes_isth = set(df[col].dropna().astype(str).str.strip().str.upper())

coinciden = genes_isth.intersection(genes_panel)
faltan = genes_isth - genes_panel
extras = genes_panel - genes_isth

print(f"Total genes en panel : {len(genes_panel)}")
print(f"Total genes en la ISTH  : {len(genes_isth)}")
print(f"Genes coincidentes      : {len(coinciden)}")

if len(faltan) == 0:
    print(" No faltan genes recomendados de la ISTH.")
else:
    print(f"Faltan {len(faltan)} genes de la ISTH:")
    print(", ".join(sorted(list(faltan))))
        
print(f"Hay {len(extras)} genes en el panel que la ISTH no contempla.")

lista_faltantes = sorted(list(faltan))

todos_genes = sorted(list(genes_panel) + lista_faltantes)

with open("panel_genes.txt", "w") as f:
    for gen in todos_genes:
        f.write(f"{gen}\n")

Total genes en panel : 202
Total genes en la ISTH  : 99
Genes coincidentes      : 98
Faltan 1 genes de la ISTH:
ERG
Hay 104 genes en el panel que la ISTH no contempla.
